# 05C2 · Development reliability training and calibration

**Verified DIV2K shards only · five-member PatchErrorNet · frozen isotonic mappings**

This notebook consumes all 12 audited Stage 05C1 shard ZIPs. It trains the frozen
six-channel ResNet-18 ensemble using only sources 0805–0856, early-stops using only
0857–0868, and fits all operational-score mappings using only 0869–0900.

The independent TESTIMAGES cohort is never loaded. This notebook cannot authorize or
perform the independent run. The long implementation is embedded and collapsed for a
clean Colab view while remaining fully auditable.


### 1. Set up the GPU runtime and Drive

Use a Colab GPU runtime. The result folder is resumable: completed ensemble members are
hash-checked and reused after an interruption.


In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

import hashlib
import json
import types
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchvision
import sklearn
from IPython.display import Image as DisplayImage, display

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/reliable-reconstruction-under-mismatch')
    CACHE_DIR = Path('/content/independent_05c2_development_cache')
else:
    PROJECT_DIR = Path.cwd()
    if PROJECT_DIR.name == 'notebooks':
        PROJECT_DIR = PROJECT_DIR.parent
    CACHE_DIR = PROJECT_DIR / 'results' / '.independent_05c2_development_cache'

assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU.'
OUTPUT_DIR = PROJECT_DIR / 'results' / 'independent_05c2_development_reliability'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0))
print('Resumable destination:', OUTPUT_DIR)


### 2. Verify and load the frozen implementation

The implementation, protocol, completed-shard registry, and current stage status are
embedded byte-for-byte. Their SHA-256 digests and the development-only barriers are
checked before any fitting begins.


In [ ]:
#@title 🔒 Embedded Stage 05C2 implementation and receipts (expand only for audit) { display-mode: "form" }
IMPLEMENTATION_SOURCE = '"""Development-only PatchErrorNet training and calibration for Experiment 05.\n\nThis module consumes only the 12 verified DIV2K development shard archives. It\nhas no loader for TESTIMAGES and cannot authorize independent inference.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport io\nimport json\nimport math\nimport os\nimport platform\nimport random\nimport re\nimport uuid\nimport zipfile\nfrom collections import defaultdict\nfrom datetime import datetime, timezone\nfrom pathlib import Path, PurePosixPath\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torchvision\nimport sklearn\nfrom sklearn.isotonic import IsotonicRegression\n\n\nEXPERIMENT_ID = "independent_05"\nSTAGE = "05C2_development_reliability_training_and_calibration"\nROLE = "development_only"\nPROTOCOL_SHA256 = "b92c6cf73e05f60dc1edb3a31d88d91623e9ae0cf63d6265e6398e335928794c"\nDEVELOPMENT_SOURCE_IDS = tuple(f"{value:04d}" for value in range(805, 901))\nFIT_SOURCE_IDS = tuple(f"{value:04d}" for value in range(805, 857))\nEARLY_STOP_SOURCE_IDS = tuple(f"{value:04d}" for value in range(857, 869))\nCALIBRATION_SOURCE_IDS = tuple(f"{value:04d}" for value in range(869, 901))\nCHAIN_IDS = (\n    "q8_b16_n2",\n    "j90_b16_n2",\n    "j75_b16_n2",\n    "j50_b16_n2",\n    "j75_b12_n2",\n    "j75_b20_n2",\n    "j75_b16_n5",\n)\nPIPELINES = ("dpir_nominal", "fbcnn_dpir_nominal")\nHEURISTIC_SCORES = (\n    "operator_spread_detail",\n    "operator_spread_rgb",\n    "image_transform_spread_detail",\n    "original_measurement_residual",\n    "reconstruction_gradient",\n)\nENSEMBLE_SCORE = "fbcnn_dpir_nominal__trained_image_only_patcherrornet_ensemble"\nENSEMBLE_VARIANCE = "fbcnn_dpir_nominal__patcherrornet_ensemble_variance"\nENSEMBLE_SEEDS = (2026092101, 2026092102, 2026092103, 2026092104, 2026092105)\nPATCH_SIZE = 16\nCONTEXT_SIZE = 64\nCONTEXT_BORDER = 32\nGRID_SIDE = 32\nPATCHES_PER_OBSERVATION = GRID_SIDE * GRID_SIDE\nSAMPLED_PATCHES_PER_OBSERVATION = 256\nBATCH_SIZE = 128\nMAXIMUM_EPOCHS = 50\nEARLY_STOPPING_PATIENCE = 5\nLEARNING_RATE = 1e-3\nWEIGHT_DECAY = 1e-4\nPRIMARY_BAD_DETAIL_THRESHOLD = 0.05\nREQUIRED_COMPACT_ARRAYS = {\n    "observation_uint8",\n    "fbcnn_dpir_nominal",\n    "detail_patch_error",\n    "target_bad_detail_0p05",\n    *{\n        f"{pipeline}__score__{score}"\n        for pipeline in PIPELINES\n        for score in HEURISTIC_SCORES\n    },\n}\n\n\ndef sha256_bytes(payload: bytes) -> str:\n    return hashlib.sha256(payload).hexdigest()\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef dump_json(path: Path, value) -> None:\n    def normalize(item):\n        if isinstance(item, dict):\n            return {str(key): normalize(val) for key, val in item.items()}\n        if isinstance(item, (list, tuple)):\n            return [normalize(val) for val in item]\n        if isinstance(item, np.generic):\n            return item.item()\n        return item\n\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + f".{uuid.uuid4().hex}.partial")\n    temporary.write_text(\n        json.dumps(normalize(value), indent=2, sort_keys=True, allow_nan=False) + "\\n",\n        encoding="utf-8",\n    )\n    temporary.replace(path)\n\n\ndef dump_csv(path: Path, frame: pd.DataFrame) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + f".{uuid.uuid4().hex}.partial")\n    frame.to_csv(temporary, index=False)\n    temporary.replace(path)\n\n\ndef source_partition(source_id: str) -> str:\n    if source_id in FIT_SOURCE_IDS:\n        return "fit"\n    if source_id in EARLY_STOP_SOURCE_IDS:\n        return "early_stop"\n    if source_id in CALIBRATION_SOURCE_IDS:\n        return "calibration"\n    raise ValueError(f"source outside frozen development partitions: {source_id}")\n\n\ndef expected_sources_for_shard(shard_index: int) -> tuple[str, ...]:\n    assert 0 <= shard_index < 12\n    start = shard_index * 8\n    return DEVELOPMENT_SOURCE_IDS[start : start + 8]\n\n\ndef validate_frozen_protocol(protocol_text: str) -> dict:\n    assert sha256_bytes(protocol_text.encode()) == PROTOCOL_SHA256, "unknown protocol bytes"\n    protocol = json.loads(protocol_text)\n    assert protocol["experiment_id"] == EXPERIMENT_ID\n    assert protocol["independent_test_run_authorized"] is False\n    assert protocol["test_results_inspected"] is False\n    partitions = protocol["development_partitions"]\n    assert partitions["trained_comparator_fit_sources"] == "0805-0856 inclusive (52 sources)"\n    assert partitions["trained_comparator_early_stop_sources"] == "0857-0868 inclusive (12 sources)"\n    assert partitions["score_calibration_sources"] == "0869-0900 inclusive (32 sources)"\n    comparator = protocol["trained_image_only_comparator"]\n    assert comparator["ensemble_members"] == len(ENSEMBLE_SEEDS) == 5\n    assert tuple(comparator["seeds"]) == ENSEMBLE_SEEDS\n    assert comparator["batch_size"] == BATCH_SIZE\n    assert comparator["maximum_epochs"] == MAXIMUM_EPOCHS\n    assert protocol["selection_scores"]["primary_bad_detail_event"] == (\n        "centre 16x16 patch detail RMSE > 0.05"\n    )\n    return protocol\n\n\ndef _archive_root_and_index(names: list[str]) -> tuple[str, int]:\n    assert names and len(names) == len(set(names))\n    assert not any(\n        name.startswith("/") or ".." in PurePosixPath(name).parts for name in names\n    )\n    roots = {name.split("/", 1)[0] for name in names}\n    assert len(roots) == 1\n    root = roots.pop()\n    match = re.fullmatch(r"independent_05c_development_shard_(\\d{2})_of_12", root)\n    assert match, root\n    return root, int(match.group(1))\n\n\ndef inspect_and_extract_shard(archive_path: Path, compact_dir: Path) -> tuple[dict, list[dict]]:\n    """Verify one full manifest and extract only the compact bundles."""\n\n    archive_path = Path(archive_path)\n    compact_dir = Path(compact_dir)\n    compact_dir.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(archive_path) as archive:\n        names = archive.namelist()\n        root, shard_index = _archive_root_and_index(names)\n\n        def read(relative: str) -> bytes:\n            return archive.read(f"{root}/{relative}")\n\n        manifest_payload = read("export_manifest.json")\n        manifest = json.loads(manifest_payload)\n        config = json.loads(read("config.json"))\n        status = json.loads(read("status.json"))\n        provenance = json.loads(read("provenance.json"))\n        expected_sources = expected_sources_for_shard(shard_index)\n        assert tuple(manifest["source_ids"]) == expected_sources\n        assert tuple(config["source_ids"]) == expected_sources\n        assert tuple(status["source_ids"]) == expected_sources\n        assert tuple(config["chain_ids"]) == CHAIN_IDS\n        assert tuple(status["chain_ids"]) == CHAIN_IDS\n        assert manifest["role"] == config["role"] == status["role"] == provenance["role"] == ROLE\n        assert manifest["test_inference_performed"] is False\n        assert status["test_inference_performed"] is False\n        assert status["test_performance_inspected"] is False\n        assert status["independent_test_run_authorized"] is False\n        assert config["test_inference_authorized"] is False\n\n        listed = {row["path"]: row for row in manifest["files"]}\n        actual = {\n            name[len(root) + 1 :]\n            for name in names\n            if name != f"{root}/export_manifest.json" and not name.endswith("/")\n        }\n        assert actual == set(listed)\n        assert len(listed) == 124\n        compact_rows = []\n        for relative, row in sorted(listed.items()):\n            payload = read(relative)\n            assert len(payload) == int(row["byte_count"]), relative\n            assert sha256_bytes(payload) == row["sha256"], relative\n            if relative.startswith("compact/") and relative.endswith(".npz"):\n                observation_id = PurePosixPath(relative).stem\n                destination = compact_dir / f"{observation_id}.npz"\n                if destination.exists():\n                    assert destination.stat().st_size == len(payload)\n                    assert sha256_file(destination) == row["sha256"]\n                else:\n                    temporary = destination.with_suffix(".npz.partial")\n                    temporary.write_bytes(payload)\n                    assert sha256_file(temporary) == row["sha256"]\n                    temporary.replace(destination)\n                source_id, chain_id = observation_id[:4], observation_id[5:]\n                assert source_id in expected_sources and chain_id in CHAIN_IDS\n                compact_rows.append(\n                    {\n                        "observation_id": observation_id,\n                        "source_id": source_id,\n                        "chain_id": chain_id,\n                        "partition": source_partition(source_id),\n                        "compact_path": str(destination),\n                        "compact_byte_count": len(payload),\n                        "compact_sha256": row["sha256"],\n                        "shard_index": shard_index,\n                    }\n                )\n        assert len(compact_rows) == 56\n        report = {\n            "shard_index": shard_index,\n            "archive_filename": archive_path.name,\n            "archive_byte_count": archive_path.stat().st_size,\n            "archive_sha256": sha256_file(archive_path),\n            "archive_root": root,\n            "export_manifest_sha256": sha256_bytes(manifest_payload),\n            "manifest_files_checked": len(listed),\n            "manifest_mismatches": 0,\n            "compact_bundles_extracted": len(compact_rows),\n            "source_ids": list(expected_sources),\n            "test_inference_performed": False,\n            "independent_test_run_authorized": False,\n        }\n        return report, compact_rows\n\n\ndef prepare_development_cache(archive_paths, cache_dir: Path) -> tuple[pd.DataFrame, dict]:\n    archive_paths = [Path(path) for path in archive_paths]\n    assert len(archive_paths) == 12, f"expected 12 shard archives, found {len(archive_paths)}"\n    cache_dir = Path(cache_dir)\n    compact_dir = cache_dir / "compact"\n    reports, rows = [], []\n    for path in sorted(archive_paths):\n        report, compact_rows = inspect_and_extract_shard(path, compact_dir)\n        print(\n            f"Verified shard {report[\'shard_index\']:02d}: "\n            f"{report[\'manifest_files_checked\']} files; {len(compact_rows)} bundles",\n            flush=True,\n        )\n        reports.append(report)\n        rows.extend(compact_rows)\n    assert sorted(row["shard_index"] for row in reports) == list(range(12))\n    frame = pd.DataFrame(rows).sort_values(["source_id", "chain_id"]).reset_index(drop=True)\n    assert len(frame) == 672\n    assert frame.observation_id.nunique() == 672\n    assert tuple(sorted(frame.source_id.unique())) == DEVELOPMENT_SOURCE_IDS\n    assert set(frame.chain_id) == set(CHAIN_IDS)\n    assert frame.groupby("source_id").size().eq(7).all()\n    partition_counts = frame.groupby("partition").source_id.nunique().to_dict()\n    assert partition_counts == {"calibration": 32, "early_stop": 12, "fit": 52}\n    dump_csv(cache_dir / "development_compact_index.csv", frame)\n    report = {\n        "experiment_id": EXPERIMENT_ID,\n        "stage": STAGE,\n        "role": ROLE,\n        "archives_verified": 12,\n        "manifest_files_checked": 12 * 124,\n        "manifest_mismatches": 0,\n        "sources": 96,\n        "observations": 672,\n        "partition_sources": partition_counts,\n        "shards": sorted(reports, key=lambda row: row["shard_index"]),\n        "test_inference_performed": False,\n        "independent_test_run_authorized": False,\n    }\n    dump_json(cache_dir / "development_cache_receipt.json", report)\n    return frame, report\n\n\ndef persist_input_receipts(index: pd.DataFrame, cache_report: dict, output_dir: Path) -> None:\n    """Persist path-free development input receipts in the exported result bundle."""\n\n    receipt_dir = Path(output_dir) / "input_receipts"\n    portable = index.drop(columns=["compact_path"]).copy()\n    dump_csv(receipt_dir / "development_compact_index.csv", portable)\n    dump_json(receipt_dir / "development_cache_receipt.json", cache_report)\n\n\ndef load_compact(path: Path) -> dict[str, np.ndarray]:\n    with np.load(path, allow_pickle=False) as stored:\n        assert set(stored.files) == REQUIRED_COMPACT_ARRAYS\n        arrays = {name: stored[name] for name in stored.files}\n    assert arrays["observation_uint8"].shape == (576, 576, 3)\n    assert arrays["observation_uint8"].dtype == np.uint8\n    assert arrays["fbcnn_dpir_nominal"].shape == (576, 576, 3)\n    assert arrays["fbcnn_dpir_nominal"].dtype == np.float32\n    assert np.isfinite(arrays["fbcnn_dpir_nominal"]).all()\n    assert 0 <= float(arrays["fbcnn_dpir_nominal"].min())\n    assert float(arrays["fbcnn_dpir_nominal"].max()) <= 1\n    target = arrays["target_bad_detail_0p05"]\n    detail_error = arrays["detail_patch_error"]\n    assert target.shape == detail_error.shape == (PATCHES_PER_OBSERVATION,)\n    assert detail_error.dtype == np.float32 and np.isfinite(detail_error).all()\n    assert float(detail_error.min()) >= 0\n    assert target.dtype == np.uint8 and set(np.unique(target)).issubset({0, 1})\n    assert np.array_equal(\n        target,\n        (np.sqrt(detail_error.astype(np.float64)) > PRIMARY_BAD_DETAIL_THRESHOLD).astype(np.uint8),\n    )\n    for pipeline in PIPELINES:\n        for score in HEURISTIC_SCORES:\n            values = arrays[f"{pipeline}__score__{score}"]\n            assert values.shape == (PATCHES_PER_OBSERVATION,)\n            assert values.dtype == np.float32 and np.isfinite(values).all()\n            assert float(values.min()) >= 0\n    return arrays\n\n\ndef _hash_u64(*parts) -> int:\n    payload = "|".join(str(part) for part in parts).encode()\n    return int.from_bytes(hashlib.sha256(payload).digest()[:8], "big", signed=False)\n\n\ndef fixed_patch_indices(\n    observation_id: str,\n    member_seed: int,\n    count: int = 256,\n    epoch: int | None = None,\n) -> np.ndarray:\n    assert 1 <= count <= PATCHES_PER_OBSERVATION\n    ranked = sorted(\n        range(PATCHES_PER_OBSERVATION),\n        key=lambda index: (\n            _hash_u64("patch", member_seed, epoch, observation_id, index),\n            index,\n        ),\n    )\n    return np.asarray(ranked[:count], dtype=np.int16)\n\n\ndef patch_context_bounds(patch_index: int) -> tuple[int, int, int, int]:\n    assert 0 <= patch_index < PATCHES_PER_OBSERVATION\n    row, column = divmod(int(patch_index), GRID_SIDE)\n    patch_top = CONTEXT_BORDER + row * PATCH_SIZE\n    patch_left = CONTEXT_BORDER + column * PATCH_SIZE\n    context_top = patch_top - (CONTEXT_SIZE - PATCH_SIZE) // 2\n    context_left = patch_left - (CONTEXT_SIZE - PATCH_SIZE) // 2\n    return context_top, context_top + CONTEXT_SIZE, context_left, context_left + CONTEXT_SIZE\n\n\ndef _d4_transform(channels: np.ndarray, code: int) -> np.ndarray:\n    assert channels.ndim == 3 and 0 <= code < 8\n    transformed = np.rot90(channels, k=code % 4, axes=(1, 2))\n    if code >= 4:\n        transformed = transformed[:, :, ::-1]\n    return np.ascontiguousarray(transformed)\n\n\ndef context_batch(\n    arrays: dict[str, np.ndarray],\n    patch_indices: np.ndarray,\n    observation_id: str,\n    member_seed: int,\n    epoch: int | None,\n) -> tuple[np.ndarray, np.ndarray]:\n    observation = arrays["observation_uint8"].astype(np.float32) / 255.0\n    reconstruction = arrays["fbcnn_dpir_nominal"].astype(np.float32, copy=False)\n    inputs, targets = [], arrays["target_bad_detail_0p05"][patch_indices].astype(np.float32)\n    for patch_index in patch_indices:\n        top, bottom, left, right = patch_context_bounds(int(patch_index))\n        six_channel = np.concatenate(\n            [observation[top:bottom, left:right], reconstruction[top:bottom, left:right]],\n            axis=2,\n        ).transpose(2, 0, 1)\n        assert six_channel.shape == (6, CONTEXT_SIZE, CONTEXT_SIZE)\n        if epoch is not None:\n            code = _hash_u64("d4", member_seed, epoch, observation_id, int(patch_index)) % 8\n            six_channel = _d4_transform(six_channel, int(code))\n        inputs.append(six_channel)\n    result = np.stack(inputs).astype(np.float32, copy=False)\n    assert np.isfinite(result).all() and 0 <= float(result.min()) <= float(result.max()) <= 1\n    return result, targets\n\n\nclass PatchErrorNet(torch.nn.Module):\n    """Frozen six-channel ResNet-18 scalar-logit architecture."""\n\n    def __init__(self):\n        super().__init__()\n        network = torchvision.models.resnet18(weights=None)\n        network.conv1 = torch.nn.Conv2d(6, 64, kernel_size=7, stride=2, padding=3, bias=False)\n        network.fc = torch.nn.Linear(network.fc.in_features, 1)\n        self.network = network\n\n    def forward(self, observation_and_reconstruction):\n        assert observation_and_reconstruction.ndim == 4\n        assert observation_and_reconstruction.shape[1] == 6\n        return self.network(observation_and_reconstruction).reshape(-1)\n\n\ndef configure_determinism(seed: int) -> None:\n    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")\n    random.seed(seed)\n    np.random.seed(seed % (2**32 - 1))\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    torch.use_deterministic_algorithms(True)\n\n\ndef runtime_environment(device) -> dict:\n    return {\n        "python": platform.python_version(),\n        "numpy": np.__version__,\n        "pandas": pd.__version__,\n        "torch": torch.__version__,\n        "torchvision": torchvision.__version__,\n        "scikit_learn": sklearn.__version__,\n        "cuda_runtime": torch.version.cuda,\n        "cudnn": torch.backends.cudnn.version(),\n        "device_type": device.type,\n        "device_name": torch.cuda.get_device_name(device) if device.type == "cuda" else "cpu",\n        "cublas_workspace_config": os.environ.get("CUBLAS_WORKSPACE_CONFIG"),\n        "deterministic_algorithms": True,\n    }\n\n\ndef _atomic_torch_save(value, path: Path) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + f".{uuid.uuid4().hex}.partial")\n    torch.save(value, temporary)\n    temporary.replace(path)\n\n\ndef training_signature(index: pd.DataFrame, member_seed: int) -> str:\n    rows = index[index.partition.isin(["fit", "early_stop"])].sort_values("observation_id")\n    payload = {\n        "protocol_sha256": PROTOCOL_SHA256,\n        "member_seed": member_seed,\n        "observation_ids": rows.observation_id.tolist(),\n        "compact_sha256": rows.compact_sha256.tolist(),\n        "batch_size": BATCH_SIZE,\n        "maximum_epochs": MAXIMUM_EPOCHS,\n        "patience": EARLY_STOPPING_PATIENCE,\n        "learning_rate": LEARNING_RATE,\n        "weight_decay": WEIGHT_DECAY,\n        "sampled_patches_per_observation": SAMPLED_PATCHES_PER_OBSERVATION,\n    }\n    return sha256_bytes(json.dumps(payload, sort_keys=True, separators=(",", ":")).encode())\n\n\ndef fit_positive_weight(index: pd.DataFrame) -> tuple[float, dict]:\n    fit_rows = index[index.partition == "fit"]\n    positives = 0\n    total = 0\n    for row in fit_rows.itertuples(index=False):\n        arrays = load_compact(Path(row.compact_path))\n        target = arrays["target_bad_detail_0p05"]\n        positives += int(target.sum())\n        total += int(target.size)\n    negatives = total - positives\n    assert positives > 0 and negatives > 0\n    return negatives / positives, {\n        "fit_patches": total,\n        "fit_positive_patches": positives,\n        "fit_negative_patches": negatives,\n        "positive_weight": negatives / positives,\n    }\n\n\ndef evaluate_member(model, index: pd.DataFrame, member_seed: int, device) -> tuple[float, dict]:\n    model.eval()\n    by_source = defaultdict(lambda: [0.0, 0])\n    validation_rows = index[index.partition == "early_stop"].sort_values("observation_id")\n    with torch.inference_mode():\n        for row in validation_rows.itertuples(index=False):\n            arrays = load_compact(Path(row.compact_path))\n            selected = fixed_patch_indices(row.observation_id, member_seed)\n            for start in range(0, len(selected), BATCH_SIZE):\n                batch_indices = selected[start : start + BATCH_SIZE]\n                inputs, targets = context_batch(\n                    arrays, batch_indices, row.observation_id, member_seed, epoch=None\n                )\n                logits = model(torch.from_numpy(inputs).to(device))\n                probabilities = torch.sigmoid(logits).cpu().numpy()\n                errors = (probabilities - targets) ** 2\n                by_source[row.source_id][0] += float(errors.sum())\n                by_source[row.source_id][1] += int(len(errors))\n    assert tuple(sorted(by_source)) == EARLY_STOP_SOURCE_IDS\n    source_brier = {source: total / count for source, (total, count) in by_source.items()}\n    macro = float(np.mean(list(source_brier.values())))\n    return macro, source_brier\n\n\ndef _load_history(path: Path) -> list[dict]:\n    if not path.exists():\n        return []\n    return pd.read_csv(path).to_dict(orient="records")\n\n\ndef train_member(\n    index: pd.DataFrame,\n    member_index: int,\n    member_seed: int,\n    work_dir: Path,\n    final_dir: Path,\n    device,\n    positive_weight: float,\n    class_receipt: dict,\n) -> dict:\n    configure_determinism(member_seed)\n    current_environment = runtime_environment(device)\n    environment_history = [current_environment]\n    signature = training_signature(index, member_seed)\n    work_dir, final_dir = Path(work_dir), Path(final_dir)\n    work_dir.mkdir(parents=True, exist_ok=True)\n    final_dir.mkdir(parents=True, exist_ok=True)\n    latest_path = work_dir / f"member_{member_index:02d}_latest.pt"\n    best_path = final_dir / f"member_{member_index:02d}_best.pt"\n    history_path = final_dir / f"member_{member_index:02d}_history.csv"\n    receipt_path = final_dir / f"member_{member_index:02d}_receipt.json"\n\n    if receipt_path.exists() and best_path.exists():\n        receipt = json.loads(receipt_path.read_text(encoding="utf-8"))\n        assert receipt["training_signature"] == signature\n        assert receipt["model_sha256"] == sha256_file(best_path)\n        print(f"Member {member_index + 1}/5 already complete; verified and reused.")\n        return receipt\n\n    model = PatchErrorNet().to(device)\n    optimizer = torch.optim.AdamW(\n        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY\n    )\n    criterion = torch.nn.BCEWithLogitsLoss(\n        pos_weight=torch.tensor([positive_weight], dtype=torch.float32, device=device)\n    )\n    history = _load_history(history_path)\n    start_epoch = 0\n    best_epoch = -1\n    best_brier = float("inf")\n    epochs_without_improvement = 0\n    if latest_path.exists():\n        checkpoint = torch.load(latest_path, map_location=device, weights_only=False)\n        assert checkpoint["training_signature"] == signature\n        model.load_state_dict(checkpoint["model_state"])\n        optimizer.load_state_dict(checkpoint["optimizer_state"])\n        start_epoch = int(checkpoint["epoch"]) + 1\n        best_epoch = int(checkpoint["best_epoch"])\n        best_brier = float(checkpoint["best_brier"])\n        epochs_without_improvement = int(checkpoint["epochs_without_improvement"])\n        history = checkpoint["history"]\n        environment_history = checkpoint.get("environment_history", [])\n        if not environment_history or environment_history[-1] != current_environment:\n            environment_history.append(current_environment)\n        dump_csv(history_path, pd.DataFrame(history))\n        print(f"Resuming member {member_index + 1}/5 at epoch {start_epoch + 1}.")\n\n    fit_rows = index[index.partition == "fit"].sort_values("observation_id").reset_index(drop=True)\n    assert fit_rows.source_id.nunique() == 52\n    for epoch in range(start_epoch, MAXIMUM_EPOCHS):\n        model.train()\n        order = np.random.default_rng(member_seed + epoch).permutation(len(fit_rows))\n        total_loss = 0.0\n        total_examples = 0\n        for row_index in order:\n            row = fit_rows.iloc[int(row_index)]\n            arrays = load_compact(Path(row.compact_path))\n            selected = fixed_patch_indices(row.observation_id, member_seed, epoch=epoch)\n            selected = selected[\n                np.random.default_rng(_hash_u64("order", member_seed, epoch, row.observation_id)).permutation(\n                    len(selected)\n                )\n            ]\n            for start in range(0, len(selected), BATCH_SIZE):\n                batch_indices = selected[start : start + BATCH_SIZE]\n                inputs, targets = context_batch(\n                    arrays, batch_indices, row.observation_id, member_seed, epoch=epoch\n                )\n                input_tensor = torch.from_numpy(inputs).to(device)\n                target_tensor = torch.from_numpy(targets).to(device)\n                optimizer.zero_grad(set_to_none=True)\n                logits = model(input_tensor)\n                loss = criterion(logits, target_tensor)\n                loss.backward()\n                optimizer.step()\n                total_loss += float(loss.detach().cpu()) * len(targets)\n                total_examples += len(targets)\n\n        validation_brier, _ = evaluate_member(model, index, member_seed, device)\n        fit_loss = total_loss / total_examples\n        improved = validation_brier < best_brier\n        if improved:\n            best_brier = validation_brier\n            best_epoch = epoch\n            epochs_without_improvement = 0\n            _atomic_torch_save(\n                {\n                    "experiment_id": EXPERIMENT_ID,\n                    "stage": STAGE,\n                    "role": ROLE,\n                    "member_index": member_index,\n                    "member_seed": member_seed,\n                    "epoch": epoch,\n                    "validation_source_macro_brier": validation_brier,\n                    "training_signature": signature,\n                    "environment_history": environment_history,\n                    "model_state": {key: value.detach().cpu() for key, value in model.state_dict().items()},\n                    "test_inference_performed": False,\n                    "independent_test_run_authorized": False,\n                },\n                best_path,\n            )\n        else:\n            epochs_without_improvement += 1\n\n        history.append(\n            {\n                "member_index": member_index,\n                "member_seed": member_seed,\n                "epoch": epoch + 1,\n                "fit_weighted_bce": fit_loss,\n                "early_stop_source_macro_brier": validation_brier,\n                "best_so_far": improved,\n            }\n        )\n        _atomic_torch_save(\n            {\n                "training_signature": signature,\n                "epoch": epoch,\n                "best_epoch": best_epoch,\n                "best_brier": best_brier,\n                "epochs_without_improvement": epochs_without_improvement,\n                "history": history,\n                "environment_history": environment_history,\n                "model_state": model.state_dict(),\n                "optimizer_state": optimizer.state_dict(),\n            },\n            latest_path,\n        )\n        dump_csv(history_path, pd.DataFrame(history))\n        print(\n            f"Member {member_index + 1}/5 epoch {epoch + 1:02d}: "\n            f"fit BCE={fit_loss:.6f}; early-stop source-macro Brier={validation_brier:.6f}",\n            flush=True,\n        )\n        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:\n            break\n\n    assert best_path.exists() and best_epoch >= 0\n    receipt = {\n        "experiment_id": EXPERIMENT_ID,\n        "stage": STAGE,\n        "role": ROLE,\n        "member_index": member_index,\n        "member_seed": member_seed,\n        "training_signature": signature,\n        "epochs_completed": len(history),\n        "best_epoch": best_epoch + 1,\n        "best_early_stop_source_macro_brier": best_brier,\n        "model_path": best_path.name,\n        "model_byte_count": best_path.stat().st_size,\n        "model_sha256": sha256_file(best_path),\n        "fit_sources": list(FIT_SOURCE_IDS),\n        "early_stop_sources": list(EARLY_STOP_SOURCE_IDS),\n        "calibration_sources_used": False,\n        "class_balance": class_receipt,\n        "environment_history": environment_history,\n        "test_inference_performed": False,\n        "independent_test_run_authorized": False,\n    }\n    dump_json(receipt_path, receipt)\n    return receipt\n\n\ndef fit_ensemble(\n    index: pd.DataFrame,\n    output_dir: Path,\n    device=None,\n    member_indices: tuple[int, ...] | list[int] | None = None,\n) -> dict:\n    output_dir = Path(output_dir)\n    work_dir = output_dir / "work_checkpoints"\n    model_dir = output_dir / "models"\n    device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))\n    assert device.type == "cuda", "PatchErrorNet fitting requires a CUDA GPU"\n    member_indices = tuple(range(5)) if member_indices is None else tuple(member_indices)\n    assert member_indices and len(member_indices) == len(set(member_indices))\n    assert all(0 <= member_index < 5 for member_index in member_indices)\n    positive_weight, class_receipt = fit_positive_weight(index)\n    for member_index in member_indices:\n        member_seed = ENSEMBLE_SEEDS[member_index]\n        train_member(\n            index,\n            member_index,\n            member_seed,\n            work_dir,\n            model_dir,\n            device,\n            positive_weight,\n            class_receipt,\n        )\n\n    receipts = []\n    for member_index in range(5):\n        receipt_path = model_dir / f"member_{member_index:02d}_receipt.json"\n        if receipt_path.exists():\n            receipt = json.loads(receipt_path.read_text(encoding="utf-8"))\n            best_path = model_dir / f"member_{member_index:02d}_best.pt"\n            assert receipt["member_seed"] == ENSEMBLE_SEEDS[member_index]\n            assert receipt["model_sha256"] == sha256_file(best_path)\n            receipts.append(receipt)\n    if len(receipts) < 5:\n        completed = sorted(int(row["member_index"]) for row in receipts)\n        return {\n            "experiment_id": EXPERIMENT_ID,\n            "stage": STAGE,\n            "role": ROLE,\n            "ensemble_complete": False,\n            "completed_member_indices": completed,\n            "pending_member_indices": sorted(set(range(5)) - set(completed)),\n            "test_inference_performed": False,\n            "independent_test_run_authorized": False,\n        }\n    assert len(receipts) == 5\n    assert all(row["calibration_sources_used"] is False for row in receipts)\n    summary = {\n        "experiment_id": EXPERIMENT_ID,\n        "stage": STAGE,\n        "role": ROLE,\n        "architecture": "ResNet-18 from scratch; six input channels; one scalar logit",\n        "ensemble_members": 5,\n        "member_seeds": list(ENSEMBLE_SEEDS),\n        "fit_source_count": 52,\n        "early_stop_source_count": 12,\n        "members": receipts,\n        "ensemble_complete": True,\n        "comparator_fitted": True,\n        "calibration_fitted": False,\n        "test_inference_performed": False,\n        "independent_test_run_authorized": False,\n    }\n    dump_json(output_dir / "ensemble_training_receipt.json", summary)\n    return summary\n\n\ndef load_frozen_ensemble(output_dir: Path, device) -> list:\n    models = []\n    for member_index, member_seed in enumerate(ENSEMBLE_SEEDS):\n        path = Path(output_dir) / "models" / f"member_{member_index:02d}_best.pt"\n        checkpoint = torch.load(path, map_location=device, weights_only=False)\n        assert checkpoint["member_seed"] == member_seed\n        assert checkpoint["test_inference_performed"] is False\n        model = PatchErrorNet().to(device)\n        model.load_state_dict(checkpoint["model_state"], strict=True)\n        model.eval()\n        models.append(model)\n    return models\n\n\ndef ensemble_probabilities(arrays: dict[str, np.ndarray], observation_id: str, models, device):\n    per_member = []\n    all_indices = np.arange(PATCHES_PER_OBSERVATION, dtype=np.int16)\n    with torch.inference_mode():\n        for model in models:\n            member_values = []\n            for start in range(0, PATCHES_PER_OBSERVATION, BATCH_SIZE):\n                batch_indices = all_indices[start : start + BATCH_SIZE]\n                inputs, _ = context_batch(\n                    arrays, batch_indices, observation_id, member_seed=0, epoch=None\n                )\n                probabilities = torch.sigmoid(model(torch.from_numpy(inputs).to(device)))\n                member_values.append(probabilities.cpu().numpy().astype(np.float32))\n            per_member.append(np.concatenate(member_values))\n    matrix = np.stack(per_member)\n    assert matrix.shape == (5, PATCHES_PER_OBSERVATION)\n    return matrix.mean(axis=0).astype(np.float32), matrix.var(axis=0).astype(np.float32)\n\n\ndef _save_npz_atomic(path: Path, arrays: dict[str, np.ndarray]) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + f".{uuid.uuid4().hex}.partial")\n    with temporary.open("wb") as stream:\n        np.savez_compressed(stream, **arrays)\n    temporary.replace(path)\n\n\ndef collect_calibration_predictions(index: pd.DataFrame, output_dir: Path, device=None) -> dict:\n    output_dir = Path(output_dir)\n    prediction_path = output_dir / "calibration_predictions.npz"\n    receipt_path = output_dir / "calibration_prediction_receipt.json"\n    if prediction_path.exists() and receipt_path.exists():\n        receipt = json.loads(receipt_path.read_text(encoding="utf-8"))\n        assert receipt["sha256"] == sha256_file(prediction_path)\n        return receipt\n\n    device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))\n    assert device.type == "cuda", "ensemble calibration inference requires a CUDA GPU"\n    models = load_frozen_ensemble(output_dir, device)\n    calibration_rows = index[index.partition == "calibration"].sort_values("observation_id")\n    assert calibration_rows.source_id.nunique() == 32 and len(calibration_rows) == 224\n    collected = defaultdict(list)\n    for observation_number, row in enumerate(calibration_rows.itertuples(index=False), start=1):\n        arrays = load_compact(Path(row.compact_path))\n        mean_probability, probability_variance = ensemble_probabilities(\n            arrays, row.observation_id, models, device\n        )\n        collected["source_id"].append(\n            np.full(PATCHES_PER_OBSERVATION, int(row.source_id), dtype=np.int16)\n        )\n        collected["chain_index"].append(\n            np.full(PATCHES_PER_OBSERVATION, CHAIN_IDS.index(row.chain_id), dtype=np.uint8)\n        )\n        collected["patch_index"].append(\n            np.arange(PATCHES_PER_OBSERVATION, dtype=np.int16)\n        )\n        collected["target_bad_detail_0p05"].append(\n            arrays["target_bad_detail_0p05"].astype(np.uint8)\n        )\n        for pipeline in PIPELINES:\n            for score in HEURISTIC_SCORES:\n                key = f"{pipeline}__score__{score}"\n                collected[key].append(arrays[key].astype(np.float32))\n        collected[ENSEMBLE_SCORE].append(mean_probability)\n        collected[ENSEMBLE_VARIANCE].append(probability_variance)\n        print(\n            f"Calibration inference [{observation_number}/224] {row.observation_id}",\n            flush=True,\n        )\n    arrays = {key: np.concatenate(values) for key, values in collected.items()}\n    expected = 32 * 7 * PATCHES_PER_OBSERVATION\n    assert {len(value) for value in arrays.values()} == {expected}\n    _save_npz_atomic(prediction_path, arrays)\n    receipt = {\n        "experiment_id": EXPERIMENT_ID,\n        "stage": STAGE,\n        "role": ROLE,\n        "path": prediction_path.name,\n        "byte_count": prediction_path.stat().st_size,\n        "sha256": sha256_file(prediction_path),\n        "sources": 32,\n        "observations": 224,\n        "patch_rows": expected,\n        "operational_scores": 11,\n        "ensemble_members": 5,\n        "test_inference_performed": False,\n        "independent_test_run_authorized": False,\n    }\n    dump_json(receipt_path, receipt)\n    return receipt\n\n\ndef _source_equal_weights(source_ids: np.ndarray) -> np.ndarray:\n    unique, counts = np.unique(source_ids, return_counts=True)\n    count_map = dict(zip(unique.tolist(), counts.tolist()))\n    weights = np.asarray([1.0 / count_map[int(source)] for source in source_ids], dtype=np.float64)\n    for source in unique:\n        assert math.isclose(float(weights[source_ids == source].sum()), 1.0, abs_tol=1e-12)\n    return weights\n\n\ndef source_macro_brier(probability, target, source_ids) -> float:\n    rows = []\n    for source in np.unique(source_ids):\n        selected = source_ids == source\n        rows.append(float(np.mean((probability[selected] - target[selected]) ** 2)))\n    return float(np.mean(rows))\n\n\ndef equal_mass_ece(probability, target, weights, bins: int = 10) -> float:\n    order = np.argsort(probability, kind="mergesort")\n    p, y, w = probability[order], target[order], weights[order]\n    cumulative = np.cumsum(w) - 0.5 * w\n    assignments = np.minimum((cumulative / w.sum() * bins).astype(int), bins - 1)\n    result = 0.0\n    for bin_index in range(bins):\n        selected = assignments == bin_index\n        if not selected.any():\n            continue\n        mass = float(w[selected].sum())\n        confidence = float(np.average(p[selected], weights=w[selected]))\n        frequency = float(np.average(y[selected], weights=w[selected]))\n        result += mass / float(w.sum()) * abs(confidence - frequency)\n    return result\n\n\ndef apply_isotonic_mapping(values: np.ndarray, mapping: dict) -> np.ndarray:\n    thresholds = np.asarray(mapping["x_thresholds"], dtype=np.float64)\n    outputs = np.asarray(mapping["y_thresholds"], dtype=np.float64)\n    oriented = np.asarray(values, dtype=np.float64) * float(mapping["orientation"])\n    return np.interp(oriented, thresholds, outputs, left=outputs[0], right=outputs[-1])\n\n\ndef fit_calibration_mappings(output_dir: Path) -> dict:\n    output_dir = Path(output_dir)\n    prediction_path = output_dir / "calibration_predictions.npz"\n    with np.load(prediction_path, allow_pickle=False) as stored:\n        arrays = {key: stored[key] for key in stored.files}\n    source_ids = arrays["source_id"].astype(np.int16)\n    target = arrays["target_bad_detail_0p05"].astype(np.float64)\n    assert tuple(f"{value:04d}" for value in sorted(np.unique(source_ids))) == CALIBRATION_SOURCE_IDS\n    weights = _source_equal_weights(source_ids)\n    score_keys = [\n        f"{pipeline}__score__{score}"\n        for pipeline in PIPELINES\n        for score in HEURISTIC_SCORES\n    ] + [ENSEMBLE_SCORE]\n    mappings, diagnostics = {}, []\n    for score_key in score_keys:\n        values = arrays[score_key].astype(np.float64)\n        assert np.isfinite(values).all()\n        orientation = 1.0\n        oriented = values * orientation\n        estimator = IsotonicRegression(\n            increasing=True, out_of_bounds="clip", y_min=0.0, y_max=1.0\n        )\n        estimator.fit(oriented, target, sample_weight=weights)\n        mapping = {\n            "score": score_key,\n            "orientation": orientation,\n            "orientation_rule": "Frozen score semantics: larger values indicate greater risk.",\n            "x_thresholds": estimator.X_thresholds_.astype(float).tolist(),\n            "y_thresholds": estimator.y_thresholds_.astype(float).tolist(),\n            "out_of_bounds": "clip",\n        }\n        calibrated = apply_isotonic_mapping(values, mapping)\n        assert np.isfinite(calibrated).all()\n        assert 0 <= float(calibrated.min()) <= float(calibrated.max()) <= 1\n        mappings[score_key] = mapping\n        diagnostics.append(\n            {\n                "score": score_key,\n                "calibration_sources": 32,\n                "calibration_patch_rows": len(target),\n                "mapping_knots": len(mapping["x_thresholds"]),\n                "source_macro_brier_in_sample": source_macro_brier(\n                    calibrated, target, source_ids\n                ),\n                "equal_mass_ece_10bin_in_sample": equal_mass_ece(\n                    calibrated, target, weights, bins=10\n                ),\n                "calibration_in_the_large_in_sample": float(\n                    np.average(target - calibrated, weights=weights)\n                ),\n            }\n        )\n    artifact = {\n        "schema_version": "1.0.0",\n        "experiment_id": EXPERIMENT_ID,\n        "stage": STAGE,\n        "role": ROLE,\n        "protocol_sha256": PROTOCOL_SHA256,\n        "method": "isotonic_regression",\n        "fit_partition": "DIV2K 0869-0900 only",\n        "weighting": "Each calibration source has equal total weight.",\n        "target": "centre 16x16 patch detail RMSE > 0.05",\n        "mappings": mappings,\n        "claim_boundary": (\n            "These are development-fitted mappings, not independent calibration results "\n            "and not posterior-probability claims."\n        ),\n        "test_inference_performed": False,\n        "independent_test_run_authorized": False,\n    }\n    dump_json(output_dir / "calibration_mappings.json", artifact)\n    pd.DataFrame(diagnostics).to_csv(output_dir / "calibration_fit_diagnostics.csv", index=False)\n    receipt = {\n        "experiment_id": EXPERIMENT_ID,\n        "stage": STAGE,\n        "role": ROLE,\n        "calibration_fitted": True,\n        "mapping_count": len(mappings),\n        "calibration_source_count": 32,\n        "calibration_patch_rows": len(target),\n        "mappings_sha256": sha256_file(output_dir / "calibration_mappings.json"),\n        "diagnostics_sha256": sha256_file(output_dir / "calibration_fit_diagnostics.csv"),\n        "test_inference_performed": False,\n        "independent_test_run_authorized": False,\n    }\n    dump_json(output_dir / "calibration_receipt.json", receipt)\n    return receipt\n\n\ndef build_diagnostic_figures(output_dir: Path) -> list[str]:\n    import matplotlib.pyplot as plt\n\n    output_dir = Path(output_dir)\n    figure_dir = output_dir / "figures"\n    figure_dir.mkdir(parents=True, exist_ok=True)\n    histories = []\n    for member_index in range(5):\n        history = pd.read_csv(output_dir / "models" / f"member_{member_index:02d}_history.csv")\n        histories.append(history)\n    history = pd.concat(histories, ignore_index=True)\n    plt.figure(figsize=(8.5, 5.0))\n    colors = ["#1f4e79", "#c58b16", "#d55e00", "#6b7d2a", "#b23a6f"]\n    for member_index, color in enumerate(colors):\n        rows = history[history.member_index == member_index]\n        plt.plot(\n            rows.epoch,\n            rows.early_stop_source_macro_brier,\n            marker="o",\n            markersize=3,\n            linewidth=1.5,\n            color=color,\n            label=f"Member {member_index + 1}",\n        )\n    plt.xlabel("Epoch")\n    plt.ylabel("Source-macro Brier score")\n    plt.title("PatchErrorNet early-stop performance (DIV2K 0857–0868)")\n    plt.grid(axis="y", color="#dddddd", linewidth=0.8)\n    plt.legend(ncol=3, frameon=False)\n    plt.tight_layout()\n    history_path = figure_dir / "patcherrornet_early_stop_history.png"\n    plt.savefig(history_path, dpi=180)\n    plt.close()\n\n    with np.load(output_dir / "calibration_predictions.npz", allow_pickle=False) as stored:\n        arrays = {key: stored[key] for key in stored.files}\n    mappings = json.loads((output_dir / "calibration_mappings.json").read_text())["mappings"]\n    target = arrays["target_bad_detail_0p05"].astype(float)\n    weights = _source_equal_weights(arrays["source_id"])\n    score_keys = list(mappings)\n    fig, axes = plt.subplots(3, 4, figsize=(13, 9), sharex=True, sharey=True)\n    axes = axes.ravel()\n    for axis, score_key in zip(axes, score_keys):\n        probability = apply_isotonic_mapping(arrays[score_key], mappings[score_key])\n        order = np.argsort(probability, kind="mergesort")\n        p, y, w = probability[order], target[order], weights[order]\n        cumulative = np.cumsum(w) - 0.5 * w\n        assignments = np.minimum((cumulative / w.sum() * 10).astype(int), 9)\n        xs, ys = [], []\n        for bin_index in range(10):\n            selected = assignments == bin_index\n            if selected.any():\n                xs.append(float(np.average(p[selected], weights=w[selected])))\n                ys.append(float(np.average(y[selected], weights=w[selected])))\n        axis.plot([0, 1], [0, 1], color="#444444", linestyle="--", linewidth=1)\n        axis.plot(xs, ys, marker="o", color="#1f4e79", linewidth=1.5)\n        axis.set_title(score_key.replace("__score__", " · ").replace("_", " "), fontsize=8)\n        axis.grid(color="#e5e5e5", linewidth=0.6)\n    for axis in axes[len(score_keys) :]:\n        axis.axis("off")\n    fig.supxlabel("Mapped probability")\n    fig.supylabel("Observed bad-detail rate")\n    fig.suptitle("Development calibration fit diagnostics (DIV2K 0869–0900)")\n    fig.tight_layout(rect=(0, 0, 1, 0.96))\n    calibration_path = figure_dir / "development_calibration_reliability.png"\n    fig.savefig(calibration_path, dpi=180)\n    plt.close(fig)\n    return [str(history_path), str(calibration_path)]\n\n\ndef finalize_development_bundle(output_dir: Path) -> dict:\n    output_dir = Path(output_dir)\n    ensemble = json.loads((output_dir / "ensemble_training_receipt.json").read_text())\n    calibration = json.loads((output_dir / "calibration_receipt.json").read_text())\n    assert ensemble["ensemble_members"] == 5 and ensemble["comparator_fitted"] is True\n    assert calibration["mapping_count"] == 11 and calibration["calibration_fitted"] is True\n    status = {\n        "experiment_id": EXPERIMENT_ID,\n        "stage": STAGE,\n        "role": ROLE,\n        "status": "passed_development_reliability_fit",\n        "comparator_fitted": True,\n        "ensemble_members": 5,\n        "calibration_fitted": True,\n        "calibration_mappings": 11,\n        "fit_sources": 52,\n        "early_stop_sources": 12,\n        "calibration_sources": 32,\n        "test_inference_performed": False,\n        "test_performance_inspected": False,\n        "independent_test_run_authorized": False,\n        "updated_at_utc": datetime.now(timezone.utc).isoformat(),\n    }\n    dump_json(output_dir / "status.json", status)\n    checks = {\n        "protocol_sha256": PROTOCOL_SHA256,\n        "ensemble_members": 5,\n        "member_model_files": 5,\n        "member_history_files": 5,\n        "calibration_mapping_count": 11,\n        "partitions_disjoint": not (\n            set(FIT_SOURCE_IDS) & set(EARLY_STOP_SOURCE_IDS)\n            or set(FIT_SOURCE_IDS) & set(CALIBRATION_SOURCE_IDS)\n            or set(EARLY_STOP_SOURCE_IDS) & set(CALIBRATION_SOURCE_IDS)\n        ),\n        "development_source_union": len(\n            set(FIT_SOURCE_IDS) | set(EARLY_STOP_SOURCE_IDS) | set(CALIBRATION_SOURCE_IDS)\n        ),\n        "test_loader_present": False,\n        "test_inference_performed": False,\n        "independent_test_run_authorized": False,\n    }\n    assert checks["partitions_disjoint"] is True\n    assert checks["development_source_union"] == 96\n    dump_json(output_dir / "checks.json", checks)\n\n    excluded = {"export_manifest.json"}\n    manifest = []\n    for path in sorted(output_dir.rglob("*")):\n        if (\n            not path.is_file()\n            or path.name in excluded\n            or ".partial" in path.name\n            or "work_checkpoints" in path.parts\n        ):\n            continue\n        manifest.append(\n            {\n                "path": str(path.relative_to(output_dir)),\n                "byte_count": path.stat().st_size,\n                "sha256": sha256_file(path),\n            }\n        )\n    dump_json(\n        output_dir / "export_manifest.json",\n        {\n            "experiment_id": EXPERIMENT_ID,\n            "stage": STAGE,\n            "role": ROLE,\n            "test_inference_performed": False,\n            "independent_test_run_authorized": False,\n            "files": manifest,\n        },\n    )\n    for row in manifest:\n        path = output_dir / row["path"]\n        assert path.stat().st_size == row["byte_count"]\n        assert sha256_file(path) == row["sha256"]\n    archive_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")\n    archive_path = output_dir.parent / f"{output_dir.name}_{archive_stamp}.zip"\n    temporary_archive = archive_path.with_suffix(f".{uuid.uuid4().hex}.partial")\n    with zipfile.ZipFile(\n        temporary_archive, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6\n    ) as archive:\n        for row in manifest:\n            archive.write(output_dir / row["path"], arcname=row["path"])\n        archive.write(output_dir / "export_manifest.json", arcname="export_manifest.json")\n    temporary_archive.replace(archive_path)\n    return {\n        "status": status["status"],\n        "output_dir": str(output_dir),\n        "archive_path": str(archive_path),\n        "archive_byte_count": archive_path.stat().st_size,\n        "archive_sha256": sha256_file(archive_path),\n        "manifest_files": len(manifest),\n        "test_inference_performed": False,\n        "independent_test_run_authorized": False,\n    }\n\n\ndef static_self_check() -> dict:\n    assert len(DEVELOPMENT_SOURCE_IDS) == 96\n    assert len(FIT_SOURCE_IDS) == 52\n    assert len(EARLY_STOP_SOURCE_IDS) == 12\n    assert len(CALIBRATION_SOURCE_IDS) == 32\n    assert not set(FIT_SOURCE_IDS) & set(EARLY_STOP_SOURCE_IDS)\n    assert not set(FIT_SOURCE_IDS) & set(CALIBRATION_SOURCE_IDS)\n    assert not set(EARLY_STOP_SOURCE_IDS) & set(CALIBRATION_SOURCE_IDS)\n    assert expected_sources_for_shard(0) == tuple(f"{value:04d}" for value in range(805, 813))\n    assert expected_sources_for_shard(11) == tuple(f"{value:04d}" for value in range(893, 901))\n    assert patch_context_bounds(0) == (8, 72, 8, 72)\n    assert patch_context_bounds(1023) == (504, 568, 504, 568)\n    selected = fixed_patch_indices("0805_j75_b16_n2", ENSEMBLE_SEEDS[0], epoch=0)\n    assert selected.shape == (256,) and len(np.unique(selected)) == 256\n    assert not np.array_equal(\n        selected,\n        fixed_patch_indices("0805_j75_b16_n2", ENSEMBLE_SEEDS[0], epoch=1),\n    )\n    return {\n        "development_sources": 96,\n        "fit_sources": 52,\n        "early_stop_sources": 12,\n        "calibration_sources": 32,\n        "ensemble_members": 5,\n        "calibration_mappings": 11,\n        "test_loader_present": False,\n        "test_inference_performed": False,\n        "independent_test_run_authorized": False,\n    }\n'
PROTOCOL_TEXT = '{\n  "schema_version": "1.0.0",\n  "experiment_id": "independent_05",\n  "stage": "05A_protocol_freeze_and_05B_documented_overlap_audit",\n  "frozen_at_utc": "2026-09-21T22:47:41Z",\n  "parent_repository_commit": "dcb9d421f55767215063398a52b8d60d869259a2",\n  "parent_experiment": "jpeg_aware_04",\n  "role": "intermediate_independent_confirmation",\n  "test_results_inspected": false,\n  "independent_test_run_authorized": false,\n  "protocol_amendment_rule": "After this freeze, only outcome-blind engineering corrections are allowed. Each correction requires a versioned amendment that identifies the bug, affected fields, reason, author, UTC time, and whether any independent output existed. No outcome-dependent change is permitted.",\n  "claim_boundary": {\n    "independent_source_evaluation_if_completed": true,\n    "checkpoint_training_overlap_proved_absent": false,\n    "reason_overlap_not_proved": "The pinned publications and repositories name training datasets but do not provide checkpoint-level image manifests.",\n    "calibrated_reliability_if_completed": "Limited to the frozen corruptions, model, score, calibration split, and bad-detail event.",\n    "novelty_established": false,\n    "definitive_100_source_gate_satisfied": false,\n    "reason_100_source_gate_not_satisfied": "The strict primary cohort has 40 natural-image sources. A successful Experiment 05 is an independent confirmation checkpoint, not the repository\'s final >=100-source gate."\n  },\n  "sealed_primary_dataset": {\n    "name": "TESTIMAGES/SAMPLING",\n    "version": "4.0.000",\n    "publisher": "Nicola Asuni / Tecnick.com",\n    "license": "CC BY-NC-SA 4.0",\n    "archive_name": "SAMPLING_8BIT_RGB_2400x2400.tar.bz2",\n    "archive_url": "https://sourceforge.net/projects/testimages/files/SAMPLING/8BIT/RGB/SAMPLING_8BIT_RGB_2400x2400.tar.bz2/download",\n    "archive_listing_url": "https://sourceforge.net/projects/testimages/files/SAMPLING/8BIT/RGB/",\n    "documentation_url": "https://testimages.org/sampling/",\n    "expected_sources": 40,\n    "expected_format": "PNG",\n    "expected_mode": "RGB",\n    "expected_bit_depth_per_channel": 8,\n    "expected_width": 2400,\n    "expected_height": 2400,\n    "member_selection": "Use every and only PNG member whose decoded image is RGB 2400x2400 8-bit. Abort unless exactly 40 unique source files satisfy the rule.",\n    "crop_rule": "Take one deterministic 576x576 centre crop per source, then evaluate the same central 512x512 region used in Experiment 04.",\n    "archive_sha256": null,\n    "archive_hash_rule": "Compute and store SHA-256 and byte count immediately after download and before extraction. A changed download is a new dataset receipt and requires a protocol amendment before any independent inference.",\n    "source_hash_rule": "Record member path, byte count, SHA-256, decoded dimensions and mode before corruption generation.",\n    "failure_rule": "Do not replace excluded, corrupt, duplicate or unreadable sources after test processing begins. Continue only with at least 36 eligible sources; otherwise abort Experiment 05.",\n    "exposure_conclusion": "No documented training or benchmark exposure was found in the reviewed pinned FBCNN and DPIR sources. This is evidence of no documented exposure, not proof that the checkpoint creators never encountered the images."\n  },\n  "development_partitions": {\n    "dataset": "DIV2K validation",\n    "checkpoint_exposure_status": "documented training-family overlap for both pretrained components; development use only",\n    "excluded_prior_development_sources": [\n      "0801",\n      "0802",\n      "0803",\n      "0804"\n    ],\n    "trained_comparator_fit_sources": "0805-0856 inclusive (52 sources)",\n    "trained_comparator_early_stop_sources": "0857-0868 inclusive (12 sources)",\n    "score_calibration_sources": "0869-0900 inclusive (32 sources)",\n    "engineering_canary_sources": [\n      "0805",\n      "0806"\n    ],\n    "partition_rule": "Source IDs are fixed lexicographically. No source may cross fit, early-stop, calibration, or independent-test partitions.",\n    "near_duplicate_audit": "Before independent inference, compare the 40 primary files with all 96 DIV2K development files using exact SHA-256 plus 64-bit pHash and dHash. Exact matches are excluded. A pair with both pHash and dHash Hamming distance <=4 is manually adjudicated before corruption generation, without viewing model outcomes. Decisions are recorded and no source is replaced."\n  },\n  "component_lock": {\n    "dpir": {\n      "repository": "cszn/DPIR",\n      "commit": "15bca3fcc1f3cc51a1f99ccf027691e278c19354",\n      "checkpoint_url": "https://github.com/cszn/KAIR/releases/download/v1.0/drunet_color.pth",\n      "checkpoint_bytes": 130579305,\n      "checkpoint_sha256": "479abe3c5327dfd10ff54a80ec7d4098ca80752a5c9492cdff31cee430bec4b4",\n      "digest_authority": "Observed official-release download; not a publisher-signed checksum."\n    },\n    "fbcnn": {\n      "repository": "jiaxi-jiang/FBCNN",\n      "commit": "54d1831927506b3247e2d4d245abb4f4dab1a1cd",\n      "checkpoint_url": "https://github.com/jiaxi-jiang/FBCNN/releases/download/v1.0/fbcnn_color.pth",\n      "checkpoint_bytes": 287755111,\n      "checkpoint_sha256": "8b0e4ef23d59cf7ac934a342cb31a17619e4fa4a0b3374a9d78c5174312387e8",\n      "digest_authority": "Observed official-release download; not a publisher-signed checksum."\n    },\n    "parent_design_sha256": "bbeda736bb0893aa7e9ddfa83678e0d00d3004ae02e50f7cf28f3444a90618f4"\n  },\n  "simulation": {\n    "seed": 20260921,\n    "crop_size": 512,\n    "context_border": 32,\n    "clean_input_scale": "decoded 8-bit RGB divided by 255 into float64 [0,1]",\n    "boundary_condition": "periodic",\n    "noise_pairing": "One source-specific standard-normal field is reused across all chains for that source and scaled by the chain noise standard deviation.",\n    "nominal_blur_sigma": 1.0,\n    "nominal_noise_std": 0.00784313725490196,\n    "jpeg_subsampling": 0,\n    "jpeg_optimize": false,\n    "acquisition_chains": [\n      {\n        "id": "q8_b16_n2",\n        "true_blur_sigma": 1.6,\n        "noise_std": 0.00784313725490196,\n        "codec": "quantized_8bit",\n        "jpeg_quality": null,\n        "role": "uncompressed_negative_control"\n      },\n      {\n        "id": "j90_b16_n2",\n        "true_blur_sigma": 1.6,\n        "noise_std": 0.00784313725490196,\n        "codec": "jpeg",\n        "jpeg_quality": 90,\n        "role": "secondary_codec_strength"\n      },\n      {\n        "id": "j75_b16_n2",\n        "true_blur_sigma": 1.6,\n        "noise_std": 0.00784313725490196,\n        "codec": "jpeg",\n        "jpeg_quality": 75,\n        "role": "primary_anchor"\n      },\n      {\n        "id": "j50_b16_n2",\n        "true_blur_sigma": 1.6,\n        "noise_std": 0.00784313725490196,\n        "codec": "jpeg",\n        "jpeg_quality": 50,\n        "role": "secondary_codec_strength"\n      },\n      {\n        "id": "j75_b12_n2",\n        "true_blur_sigma": 1.2,\n        "noise_std": 0.00784313725490196,\n        "codec": "jpeg",\n        "jpeg_quality": 75,\n        "role": "secondary_blur_strength"\n      },\n      {\n        "id": "j75_b20_n2",\n        "true_blur_sigma": 2.0,\n        "noise_std": 0.00784313725490196,\n        "codec": "jpeg",\n        "jpeg_quality": 75,\n        "role": "secondary_blur_strength"\n      },\n      {\n        "id": "j75_b16_n5",\n        "true_blur_sigma": 1.6,\n        "noise_std": 0.0196078431372549,\n        "codec": "jpeg",\n        "jpeg_quality": 75,\n        "role": "secondary_noise_strength"\n      }\n    ]\n  },\n  "reconstruction_methods": [\n    "observed",\n    "gradient_nominal",\n    "dpir_nominal",\n    "fbcnn",\n    "fbcnn_gradient_nominal",\n    "fbcnn_dpir_nominal"\n  ],\n  "fixed_reconstruction_settings": {\n    "gradient_lambda": 0.05,\n    "dpir_iterations": 8,\n    "model_sigma_start_255": 49.0,\n    "model_sigma_end_255": 2.0,\n    "prior_tradeoff": 0.23,\n    "periodic_x8": true,\n    "fbcnn_quality_input": "automatic prediction only",\n    "fbcnn_output": "clip to [0,1] without uint8 rounding before downstream inversion"\n  },\n  "selection_scores": {\n    "operational": [\n      "operator_spread_detail",\n      "operator_spread_rgb",\n      "image_transform_spread_detail",\n      "original_measurement_residual",\n      "reconstruction_gradient",\n      "trained_image_only_patcherrornet_ensemble"\n    ],\n    "evaluation_only": [\n      "expected_random",\n      "oracle_detail_error"\n    ],\n    "operator_sigmas": [\n      0.8,\n      1.0,\n      1.2\n    ],\n    "outer_rotations_quarter_turns": [\n      0,\n      1,\n      2\n    ],\n    "patch_size": 16,\n    "coverages": [\n      0.5,\n      0.75,\n      0.9,\n      1.0\n    ],\n    "detail_operator": "D(x)=x-GaussianBlur_sigma1(x)",\n    "detail_rmse_tolerances": [\n      0.025,\n      0.05,\n      0.1\n    ],\n    "primary_bad_detail_event": "centre 16x16 patch detail RMSE > 0.05"\n  },\n  "trained_image_only_comparator": {\n    "name": "PatchErrorNet deep ensemble",\n    "information": "64x64 context from the supplied observation and FBCNN+DPIR reconstruction only; no operator variants, source ID, clean image, true blur, true noise, codec metadata or reference-derived feature",\n    "input_channels": 6,\n    "backbone": "ResNet-18 from scratch with a 6-channel first convolution and one scalar logit",\n    "target": "Whether the centre 16x16 patch has detail RMSE > 0.05",\n    "ensemble_members": 5,\n    "seeds": [\n      2026092101,\n      2026092102,\n      2026092103,\n      2026092104,\n      2026092105\n    ],\n    "optimizer": "AdamW",\n    "learning_rate": 0.001,\n    "weight_decay": 0.0001,\n    "batch_size": 128,\n    "maximum_epochs": 50,\n    "early_stopping": "Patience 5 on source-macro Brier score in sources 0857-0868; retain the lowest validation-Brier epoch.",\n    "loss": "Binary cross-entropy with positive weight estimated from fit sources only.",\n    "augmentation": "D4 rotations/reflections applied deterministically from member seed.",\n    "sampling": "At most 256 fixed hash-selected patch centres per source and acquisition chain per epoch, source-balanced.",\n    "primary_score": "Mean predicted bad-detail probability across five members.",\n    "secondary_score": "Variance of predicted bad-detail probability across five members."\n  },\n  "calibration": {\n    "fit_partition": "DIV2K 0869-0900 only",\n    "method": "Isotonic regression fitted separately for each operational score after orienting higher values to mean higher risk.",\n    "weights": "Each calibration source has equal total weight.",\n    "test_metrics": [\n      "source-macro Brier score",\n      "10-bin equal-mass expected calibration error",\n      "calibration-in-the-large",\n      "calibration slope",\n      "reliability diagram"\n    ],\n    "calibration_claim": "A score is called calibrated only for the frozen bad-detail event and acquisition family if its independent-test metrics and source-bootstrap intervals are reported. Calibration is not a posterior-probability claim."\n  },\n  "primary_hypotheses": [\n    {\n      "id": "H1_reconstruction",\n      "condition": "j75_b16_n2",\n      "unit": "source",\n      "contrast": "fbcnn_dpir_nominal minus dpir_nominal",\n      "endpoint": "mean detail MSE over the 512x512 evaluation region",\n      "direction": "negative is favorable",\n      "practical_gate": "At least 5% relative reduction versus dpir_nominal.",\n      "statistical_gate": "Holm-adjusted one-sided paired sign-flip p < 0.05 and the two-sided 95% paired source-bootstrap interval for the mean difference lies entirely below zero."\n    },\n    {\n      "id": "H2_selection",\n      "condition": "j75_b16_n2",\n      "pipeline": "fbcnn_dpir_nominal",\n      "coverage": 0.5,\n      "unit": "source",\n      "contrast": "operator_spread_detail risk minus image_transform_spread_detail risk",\n      "endpoint": "retained-patch detail MSE",\n      "direction": "negative is favorable",\n      "practical_gate": "At least 5% relative risk reduction versus image_transform_spread_detail.",\n      "statistical_gate": "Holm-adjusted one-sided paired sign-flip p < 0.05 and the two-sided 95% paired source-bootstrap interval for the mean difference lies entirely below zero."\n    }\n  ],\n  "statistical_plan": {\n    "independent_unit": "source image",\n    "patches_as_independent_replicates": false,\n    "bootstrap": "10,000 source-level paired percentile resamples using seed 20260921",\n    "randomization_test": "100,000 source-level paired sign flips using seed 20260921; use exact enumeration only if the analyzable source count makes it cheaper",\n    "multiplicity": "Holm correction across H1 and H2 only",\n    "secondary_outcomes": "Report point estimates and 95% source-bootstrap intervals. They cannot rescue a failed primary gate and are labelled exploratory unless explicitly named as key secondary.",\n    "missingness": "Report every exclusion and failed observation. Do not impute or replace sources."\n  },\n  "execution_barriers": [\n    {\n      "stage": "05A-05B",\n      "allowed": "Protocol freeze, documented exposure audit, static validation notebook.",\n      "forbidden": "Download or inspect independent reconstruction outputs, risk curves or performance summaries."\n    },\n    {\n      "stage": "05C",\n      "allowed": "Acquire and hash data; run byte/decode/duplicate checks; run engineering canary on DIV2K 0805-0806 only.",\n      "forbidden": "Run inference on TESTIMAGES or change scientific settings using canary quality outcomes."\n    },\n    {\n      "stage": "05D",\n      "allowed": "Locked full run after every readiness assertion passes; resume failed jobs without inspecting aggregate outcomes.",\n      "forbidden": "Interim test peeking, method tuning, threshold tuning, source replacement, or selective reruns based on performance."\n    },\n    {\n      "stage": "05E-05F",\n      "allowed": "One-time unsealing, calibration application, frozen statistics, reporting and pass/fail decision.",\n      "forbidden": "Changing hypotheses, comparators, exclusions or decision thresholds after unsealing."\n    }\n  ],\n  "readiness_requirements_before_independent_inference": [\n    "Archive SHA-256 and byte count receipt exists.",\n    "Exactly 40 eligible source records exist before duplicate adjudication.",\n    "All decoded sources are RGB 2400x2400 8-bit PNGs.",\n    "Exact and near-duplicate audit against all development sources is complete.",\n    "At least 36 primary sources remain eligible.",\n    "Pinned DPIR and FBCNN source and checkpoint digests match.",\n    "PatchErrorNet training and early stopping are complete without calibration/test data.",\n    "All isotonic mappings are fitted on DIV2K 0869-0900 only and serialized.",\n    "The DIV2K 0805-0806 engineering canary passes schema, determinism, range, hash, resume and independent-readback checks.",\n    "The run code verifies this protocol\'s SHA-256 and refuses an unknown protocol.",\n    "No independent performance artifact exists before the locked run starts."\n  ],\n  "reviewed_primary_sources": [\n    {\n      "title": "FBCNN official training configuration",\n      "url": "https://github.com/jiaxi-jiang/FBCNN/blob/54d1831927506b3247e2d4d245abb4f4dab1a1cd/options/train_fbcnn_color.json"\n    },\n    {\n      "title": "FBCNN official colour test script",\n      "url": "https://github.com/jiaxi-jiang/FBCNN/blob/54d1831927506b3247e2d4d245abb4f4dab1a1cd/main_test_fbcnn_color.py"\n    },\n    {\n      "title": "Towards Flexible Blind JPEG Artifacts Removal",\n      "url": "https://arxiv.org/abs/2109.14573"\n    },\n    {\n      "title": "Plug-and-Play Image Restoration with Deep Denoiser Prior",\n      "url": "https://arxiv.org/abs/2008.13751"\n    },\n    {\n      "title": "DPIR official repository",\n      "url": "https://github.com/cszn/DPIR/tree/15bca3fcc1f3cc51a1f99ccf027691e278c19354"\n    },\n    {\n      "title": "TESTIMAGES/SAMPLING documentation",\n      "url": "https://testimages.org/sampling/"\n    },\n    {\n      "title": "TESTIMAGES/SAMPLING 8-bit RGB archive listing",\n      "url": "https://sourceforge.net/projects/testimages/files/SAMPLING/8BIT/RGB/"\n    },\n    {\n      "title": "TESTIMAGES: A Large Data Archive For Display and Algorithm Testing",\n      "url": "https://doi.org/10.1080/2165347X.2015.1024298"\n    }\n  ],\n  "current_status": {\n    "protocol_frozen": true,\n    "documented_exposure_audit_complete": true,\n    "checkpoint_training_overlap_proved_absent": false,\n    "independent_dataset_downloaded_and_hashed": false,\n    "near_duplicate_audit_complete": false,\n    "trained_comparator_ready": false,\n    "calibration_mappings_ready": false,\n    "engineering_canary_passed": false,\n    "independent_test_run": false,\n    "next_stage": "05C_data_receipt_implementation_and_development_only_canary"\n  }\n}\n'
REGISTRY_TEXT = '{\n  "experiment_id": "independent_05",\n  "expected_shards": 12,\n  "independent_test_run_authorized": false,\n  "next_expected_shard_index": null,\n  "role": "development_only",\n  "schema_version": "1.0.0",\n  "shards": [\n    {\n      "archive_byte_count": 235760220,\n      "archive_root": "independent_05c_development_shard_00_of_12",\n      "archive_sha256": "4b7a011a4300417104278e673a2e61eb4a301f776a95a96259cafcecb25ea608",\n      "receipt": "experiments/independent_05/development_shard_00_readback_receipt.json",\n      "shard_index": 0,\n      "source_ids": [\n        "0805",\n        "0806",\n        "0807",\n        "0808",\n        "0809",\n        "0810",\n        "0811",\n        "0812"\n      ],\n      "status": "verified_pass",\n      "verified_at_utc": "2026-09-24T14:59:31Z"\n    },\n    {\n      "archive_byte_count": 235238703,\n      "archive_root": "independent_05c_development_shard_01_of_12",\n      "archive_sha256": "2e26fcd21b96a5a3360708f989f097c92c451360504cd318fe4e5c46ba63dc0d",\n      "findings": [\n        "received_outer_zip_repackaged_or_rewrapped"\n      ],\n      "notebook_reported_archive_sha256": "974aaa068a8d17f3ef554d0e88db4a5fe2a8d29d065a864016bb322e03b65681",\n      "outer_sha256_match": false,\n      "receipt": "experiments/independent_05/development_shard_01_readback_receipt.json",\n      "shard_index": 1,\n      "source_ids": [\n        "0813",\n        "0814",\n        "0815",\n        "0816",\n        "0817",\n        "0818",\n        "0819",\n        "0820"\n      ],\n      "status": "verified_pass_with_recorded_outer_zip_mismatch",\n      "verified_at_utc": "2026-09-24T16:54:25Z"\n    },\n    {\n      "archive_byte_count": 239773580,\n      "archive_root": "independent_05c_development_shard_02_of_12",\n      "archive_sha256": "54da48f2ee8658d5d2c8107aba54dfe6cb638cda62301585f0e929debcabb421",\n      "findings": [\n        "received_outer_zip_repackaged_or_rewrapped"\n      ],\n      "notebook_reported_archive_sha256": "e13fd2c6a9d24b0dc75d24b42274a771ba3840da150cde3ecfa1831a26506aee",\n      "outer_sha256_match": false,\n      "receipt": "experiments/independent_05/development_shard_02_readback_receipt.json",\n      "shard_index": 2,\n      "source_ids": [\n        "0821",\n        "0822",\n        "0823",\n        "0824",\n        "0825",\n        "0826",\n        "0827",\n        "0828"\n      ],\n      "status": "verified_pass_with_recorded_outer_zip_mismatch",\n      "verified_at_utc": "2026-09-24T17:23:47Z"\n    },\n    {\n      "archive_byte_count": 234302354,\n      "archive_root": "independent_05c_development_shard_03_of_12",\n      "archive_sha256": "a63d8b69082a3b00e7e485363a73f96e4860919f11bcf30c7f300fd0bb58ef79",\n      "receipt": "experiments/independent_05/development_shard_03_readback_receipt.json",\n      "shard_index": 3,\n      "source_ids": [\n        "0829",\n        "0830",\n        "0831",\n        "0832",\n        "0833",\n        "0834",\n        "0835",\n        "0836"\n      ],\n      "status": "verified_pass",\n      "verified_at_utc": "2026-09-24T18:01:21Z"\n    },\n    {\n      "archive_byte_count": 233760932,\n      "archive_root": "independent_05c_development_shard_04_of_12",\n      "archive_sha256": "50428120c6b755cb246b701d4e7fbd779addf54ab94127c86e1961e735476bd6",\n      "findings": [\n        "received_outer_zip_repackaged_or_rewrapped"\n      ],\n      "notebook_reported_archive_sha256": "77aa6243e100ccafcacb7abfa2a243870d489de9f75b3b40a414936427374fb3",\n      "outer_sha256_match": false,\n      "receipt": "experiments/independent_05/development_shard_04_readback_receipt.json",\n      "shard_index": 4,\n      "source_ids": [\n        "0837",\n        "0838",\n        "0839",\n        "0840",\n        "0841",\n        "0842",\n        "0843",\n        "0844"\n      ],\n      "status": "verified_pass_with_recorded_outer_zip_mismatch",\n      "verified_at_utc": "2026-09-24T18:26:59Z"\n    },\n    {\n      "archive_byte_count": 235125077,\n      "archive_root": "independent_05c_development_shard_05_of_12",\n      "archive_sha256": "3667e151faafd668019fa44e51e7ffc72090fc7c744047d9ae1837dcc5ca39f7",\n      "findings": [\n        "received_outer_zip_repackaged_or_rewrapped"\n      ],\n      "notebook_reported_archive_sha256": "228451c2f170d23615cde448ed55c3f37f8ab224ca401ab922d4fbecfe2a24ff",\n      "outer_sha256_match": false,\n      "receipt": "experiments/independent_05/development_shard_05_readback_receipt.json",\n      "shard_index": 5,\n      "source_ids": [\n        "0845",\n        "0846",\n        "0847",\n        "0848",\n        "0849",\n        "0850",\n        "0851",\n        "0852"\n      ],\n      "status": "verified_pass_with_recorded_outer_zip_mismatch",\n      "verified_at_utc": "2026-09-24T18:47:36Z"\n    },\n    {\n      "archive_byte_count": 240558332,\n      "archive_root": "independent_05c_development_shard_06_of_12",\n      "archive_sha256": "fa145e5ae4568ade3f23b6aca95fdf0a149ce117d721451b52d97acd8a77bee6",\n      "findings": [\n        "received_outer_zip_repackaged_or_rewrapped"\n      ],\n      "notebook_reported_archive_sha256": "e5f98c2dd138ff21b9d8793ad59d7a56a771bc71ad644c333abb7b5c849bab33",\n      "outer_sha256_match": false,\n      "receipt": "experiments/independent_05/development_shard_06_readback_receipt.json",\n      "shard_index": 6,\n      "source_ids": [\n        "0853",\n        "0854",\n        "0855",\n        "0856",\n        "0857",\n        "0858",\n        "0859",\n        "0860"\n      ],\n      "status": "verified_pass_with_recorded_outer_zip_mismatch",\n      "verified_at_utc": "2026-09-24T18:53:04Z"\n    },\n    {\n      "archive_byte_count": 231758842,\n      "archive_root": "independent_05c_development_shard_07_of_12",\n      "archive_sha256": "9115cfb0a6c01b24615ea5d1357ed17a5fd6f04eb126966a0d7d5b91d6719a5f",\n      "findings": [\n        "received_outer_zip_repackaged_or_rewrapped"\n      ],\n      "notebook_reported_archive_sha256": "e64fba9846e7a64efe9d01ca24ffa09e74d431fb8941f4502235a2f98cde193c",\n      "outer_sha256_match": false,\n      "receipt": "experiments/independent_05/development_shard_07_readback_receipt.json",\n      "shard_index": 7,\n      "source_ids": [\n        "0861",\n        "0862",\n        "0863",\n        "0864",\n        "0865",\n        "0866",\n        "0867",\n        "0868"\n      ],\n      "status": "verified_pass_with_recorded_outer_zip_mismatch",\n      "verified_at_utc": "2026-09-24T19:52:53Z"\n    },\n    {\n      "archive_byte_count": 241234683,\n      "archive_root": "independent_05c_development_shard_08_of_12",\n      "archive_sha256": "ad5224d814e094377b8c85b657b287ba1a936b5d7034a31197859dd2ccc766ee",\n      "findings": [\n        "received_outer_zip_repackaged_or_rewrapped"\n      ],\n      "notebook_reported_archive_sha256": "bdf7be260717894939ca87bc0f371221bcd503c09f3601bec15e30f534744a5b",\n      "outer_sha256_match": false,\n      "receipt": "experiments/independent_05/development_shard_08_readback_receipt.json",\n      "shard_index": 8,\n      "source_ids": [\n        "0869",\n        "0870",\n        "0871",\n        "0872",\n        "0873",\n        "0874",\n        "0875",\n        "0876"\n      ],\n      "status": "verified_pass_with_recorded_outer_zip_mismatch",\n      "verified_at_utc": "2026-09-24T19:52:53Z"\n    },\n    {\n      "archive_byte_count": 235786675,\n      "archive_root": "independent_05c_development_shard_09_of_12",\n      "archive_sha256": "dea528681ac602bdb2485ac0fa96aac0f86d1b08f92c85a0e20a302ac8e221cc",\n      "findings": [\n        "received_outer_zip_repackaged_or_rewrapped"\n      ],\n      "notebook_reported_archive_sha256": "6ab19021a8c34d339b84c7d58be33c951bfd4977389e696a54a19a3c5777c21d",\n      "outer_sha256_match": false,\n      "receipt": "experiments/independent_05/development_shard_09_readback_receipt.json",\n      "shard_index": 9,\n      "source_ids": [\n        "0877",\n        "0878",\n        "0879",\n        "0880",\n        "0881",\n        "0882",\n        "0883",\n        "0884"\n      ],\n      "status": "verified_pass_with_recorded_outer_zip_mismatch",\n      "verified_at_utc": "2026-09-24T20:19:46Z"\n    },\n    {\n      "archive_byte_count": 238455591,\n      "archive_root": "independent_05c_development_shard_10_of_12",\n      "archive_sha256": "44418f8f5b3ca3a8163b2d7f62c5c1a5fe1fa5a8b73d2089b38c29dc97909475",\n      "findings": [\n        "received_outer_zip_repackaged_or_rewrapped"\n      ],\n      "notebook_reported_archive_sha256": "5621d907b0ee72e1c2658c89b68b2579120a67d76a8d588171b41fcb8bdb180b",\n      "outer_sha256_match": false,\n      "receipt": "experiments/independent_05/development_shard_10_readback_receipt.json",\n      "shard_index": 10,\n      "source_ids": [\n        "0885",\n        "0886",\n        "0887",\n        "0888",\n        "0889",\n        "0890",\n        "0891",\n        "0892"\n      ],\n      "status": "verified_pass_with_recorded_outer_zip_mismatch",\n      "verified_at_utc": "2026-09-24T20:19:46Z"\n    },\n    {\n      "archive_byte_count": 235138720,\n      "archive_root": "independent_05c_development_shard_11_of_12",\n      "archive_sha256": "6e1cfebfe5a140b8dbfa78d3b2004ac0b8eb21e6732e9f59f95442b021ec655e",\n      "findings": [\n        "received_outer_zip_repackaged_or_rewrapped"\n      ],\n      "notebook_reported_archive_sha256": "9d6920327287ba37c1d0566b431b9fefc609dbf83ad4c5cf9be73a37ebd151d2",\n      "outer_sha256_match": false,\n      "receipt": "experiments/independent_05/development_shard_11_readback_receipt.json",\n      "shard_index": 11,\n      "source_ids": [\n        "0893",\n        "0894",\n        "0895",\n        "0896",\n        "0897",\n        "0898",\n        "0899",\n        "0900"\n      ],\n      "status": "verified_pass_with_recorded_outer_zip_mismatch",\n      "verified_at_utc": "2026-09-24T20:19:46Z"\n    }\n  ],\n  "stage": "05C_development_generation",\n  "test_inference_performed": false,\n  "test_performance_inspected": false,\n  "updated_at_utc": "2026-09-24T20:19:46Z",\n  "verified_shards": 12\n}\n\n'
STAGE_STATUS_TEXT = '{\n  "blockers": [],\n  "calibration_fitted": false,\n  "canary_readback_receipt": {\n    "path": "experiments/independent_05/canary_readback_receipt.json",\n    "status": "pass_with_recorded_outcome_blind_metadata_corrections"\n  },\n  "canary_readback_verified": true,\n  "canary_inputs_ready": true,\n  "canary_notebook_prepared": true,\n  "canary_notebook_execution": "passed_cuda_colab_readback_verified",\n  "comparator_fitted": false,\n  "data_receipt": {\n    "byte_count": 54539,\n    "path": "experiments/independent_05/data_receipt.json",\n    "sha256": "ad48eb7a65b043173f1012035b4a4c95d0299420614311bdfc6611ef840169cb"\n  },\n  "dataset_downloaded_and_hashed": true,\n  "development_generation_notebook_prepared": true,\n  "development_reliability_notebook_prepared": true,\n  "development_generation_shards_completed": 12,\n  "development_generation_shards_expected": 12,\n  "development_canary_passed": true,\n  "experiment_id": "independent_05",\n  "implementation_ready": true,\n  "independent_test_run_authorized": false,\n  "near_duplicate_audit_complete": true,\n  "next_permitted_action": "Run Notebook 05C2 on the 12 verified development shard ZIPs, return its result ZIP and executed notebook for readback, and keep independent test inference unauthorized.",\n  "protocol_frozen": true,\n  "schema_version": "1.0.0",\n  "stage": "05C",\n  "test_archive_publisher_digest_match": true,\n  "test_inference_performed": false,\n  "test_performance_inspected": false,\n  "test_source_decode_check_complete": true,\n  "updated_at_utc": "2026-09-24T20:52:41Z",\n  "development_shard_registry": {\n    "path": "experiments/independent_05/development_shard_registry.json",\n    "verified_shards": 12\n  },\n  "latest_development_shard_receipt": {\n    "path": "experiments/independent_05/development_shard_11_readback_receipt.json",\n    "shard_index": 11,\n    "status": "pass_with_recorded_outer_zip_mismatch"\n  },\n  "development_generation_observations_verified": 672,\n  "development_generation_sources_verified": 96,\n  "development_generation_complete": true\n}\n'
EXPECTED_DIGESTS = {'implementation': 'a00f119f87c075e6cc9b8297e0a28536ab8c5d27e513690475279980592ee8ef', 'protocol': 'b92c6cf73e05f60dc1edb3a31d88d91623e9ae0cf63d6265e6398e335928794c', 'registry': 'df288556f8d009999b2a2264e635eba7673fecc1704634d97290aa8f47cd9c5a', 'stage_status': 'd65df363f5a0d81804735ce1a9831605e3b3dc9b207e302c4e74175ec214c2ae'}

assert hashlib.sha256(IMPLEMENTATION_SOURCE.encode()).hexdigest() == EXPECTED_DIGESTS['implementation']
assert hashlib.sha256(PROTOCOL_TEXT.encode()).hexdigest() == EXPECTED_DIGESTS['protocol']
assert hashlib.sha256(REGISTRY_TEXT.encode()).hexdigest() == EXPECTED_DIGESTS['registry']
assert hashlib.sha256(STAGE_STATUS_TEXT.encode()).hexdigest() == EXPECTED_DIGESTS['stage_status']

REGISTRY_05C = json.loads(REGISTRY_TEXT)
STAGE_STATUS_05C = json.loads(STAGE_STATUS_TEXT)
assert REGISTRY_05C['verified_shards'] == 12
assert REGISTRY_05C['next_expected_shard_index'] is None
assert REGISTRY_05C['independent_test_run_authorized'] is False
assert STAGE_STATUS_05C['development_generation_complete'] is True
assert STAGE_STATUS_05C['development_generation_shards_completed'] == 12
assert STAGE_STATUS_05C['comparator_fitted'] is False
assert STAGE_STATUS_05C['calibration_fitted'] is False
assert STAGE_STATUS_05C['independent_test_run_authorized'] is False
assert STAGE_STATUS_05C['test_inference_performed'] is False

RT05 = types.ModuleType('independent_05_reliability_training_embedded')
exec(compile(IMPLEMENTATION_SOURCE, '<independent_05_reliability_training_embedded>', 'exec'), RT05.__dict__)
RT05.validate_frozen_protocol(PROTOCOL_TEXT)
print(json.dumps(RT05.static_self_check(), indent=2))


### 3. Locate and independently verify all 12 shard archives

The default archive folder is the `results` folder used by Notebook 05C1. Every one of
the 1,488 manifest-listed files is byte-counted and SHA-256 checked. Only the 672 compact
development bundles are extracted into the runtime cache.


In [ ]:
SHARD_ARCHIVE_DIR_TEXT = ''  #@param {type:"string"}
SHARD_ARCHIVE_DIR = (
    Path(SHARD_ARCHIVE_DIR_TEXT) if SHARD_ARCHIVE_DIR_TEXT.strip()
    else PROJECT_DIR / 'results'
)
archive_candidates = sorted(
    SHARD_ARCHIVE_DIR.glob('independent_05c_development_shard_*_of_12*.zip')
)
assert archive_candidates, f'No Stage 05C1 shard ZIPs found in {SHARD_ARCHIVE_DIR}'

by_index = {}
for path in archive_candidates:
    with zipfile.ZipFile(path) as archive:
        roots = {name.split('/', 1)[0] for name in archive.namelist() if '/' in name}
        matching = [root for root in roots if root.startswith('independent_05c_development_shard_')]
        assert len(matching) == 1, f'Unexpected archive root in {path.name}'
        shard_index = int(matching[0].split('_')[-3])
    by_index.setdefault(shard_index, []).append(path)

duplicates = {index: paths for index, paths in by_index.items() if len(paths) != 1}
assert not duplicates, (
    'Expected exactly one ZIP per shard index. Move duplicate reruns out of the selected '
    f'folder and rerun this cell: {duplicates}'
)
assert sorted(by_index) == list(range(12)), f'Missing shard indices: {sorted(set(range(12)) - set(by_index))}'
SHARD_ARCHIVES = [by_index[index][0] for index in range(12)]

DEVELOPMENT_INDEX, CACHE_RECEIPT = RT05.prepare_development_cache(
    SHARD_ARCHIVES, CACHE_DIR
)
RT05.persist_input_receipts(DEVELOPMENT_INDEX, CACHE_RECEIPT, OUTPUT_DIR)
print(json.dumps({
    'archives_verified': CACHE_RECEIPT['archives_verified'],
    'manifest_files_checked': CACHE_RECEIPT['manifest_files_checked'],
    'sources': CACHE_RECEIPT['sources'],
    'observations': CACHE_RECEIPT['observations'],
    'partition_sources': CACHE_RECEIPT['partition_sources'],
}, indent=2))


### 4. Train or resume the five-member PatchErrorNet ensemble

Default: train members `0,1,2,3,4` sequentially in this runtime.

For parallel Colabs, duplicate this notebook and give each runtime disjoint member indices
(for example `0,1`, `2,3`, and `4`). They may share the same Drive result folder because
member filenames do not overlap. Once all five finish, rerun this cell in any one runtime;
completed members are verified and reused.


In [ ]:
MEMBER_INDICES_TEXT = '0,1,2,3,4'  #@param {type:"string"}
MEMBER_INDICES = tuple(int(value.strip()) for value in MEMBER_INDICES_TEXT.split(',') if value.strip())
assert MEMBER_INDICES and len(MEMBER_INDICES) == len(set(MEMBER_INDICES))
assert all(0 <= value < 5 for value in MEMBER_INDICES)

ENSEMBLE_RECEIPT = RT05.fit_ensemble(
    DEVELOPMENT_INDEX,
    OUTPUT_DIR,
    device='cuda',
    member_indices=MEMBER_INDICES,
)
ENSEMBLE_READY = ENSEMBLE_RECEIPT['ensemble_complete']
print(json.dumps(ENSEMBLE_RECEIPT, indent=2))
if not ENSEMBLE_READY:
    print('This member subset is safely complete. Pending members:', ENSEMBLE_RECEIPT['pending_member_indices'])
    print('After all parallel runtimes finish, rerun this cell in any one notebook.')


### 5. Freeze the ensemble and fit source-separated calibration mappings

This runs only after all five model receipts are present. It evaluates the frozen ensemble
on calibration sources 0869–0900 and fits 11 isotonic mappings: five heuristic scores for
each of two pipelines, plus the PatchErrorNet ensemble mean. Every calibration source has
equal total fitting weight.


In [ ]:
if ENSEMBLE_READY:
    CALIBRATION_PREDICTION_RECEIPT = RT05.collect_calibration_predictions(
        DEVELOPMENT_INDEX, OUTPUT_DIR, device='cuda'
    )
    CALIBRATION_RECEIPT = RT05.fit_calibration_mappings(OUTPUT_DIR)
    FIGURE_PATHS = RT05.build_diagnostic_figures(OUTPUT_DIR)
    print(json.dumps(CALIBRATION_PREDICTION_RECEIPT, indent=2))
    print(json.dumps(CALIBRATION_RECEIPT, indent=2))
    for figure_path in FIGURE_PATHS:
        display(DisplayImage(filename=figure_path))
else:
    CALIBRATION_RECEIPT = None
    print('Calibration skipped until all five ensemble members are complete.')


### 6. Audit and package the development reliability bundle

The final ZIP contains frozen model checkpoints, histories, calibration predictions,
mappings, diagnostics, figures, and cryptographic receipts. Mutable resume checkpoints are
deliberately excluded. Upload the ZIP and the executed notebook for independent readback.


In [ ]:
if ENSEMBLE_READY and CALIBRATION_RECEIPT is not None:
    FINAL_RESULT = RT05.finalize_development_bundle(OUTPUT_DIR)
    print(json.dumps(FINAL_RESULT, indent=2))
    print('UPLOAD THIS RESULT ZIP:', FINAL_RESULT['archive_path'])
    print('ALSO DOWNLOAD THIS EXECUTED NOTEBOOK from Colab: File > Download > .ipynb')
    print('Independent test run authorized: false')
else:
    FINAL_RESULT = None
    print('Packaging skipped until ensemble fitting and calibration both finish.')


## Handoff

Return two files after the green final cell:

1. `independent_05c2_development_reliability_<timestamp>.zip`
2. the executed `05C2_Development_Reliability_Training_and_Calibration.ipynb`

Successful completion establishes that the trained comparator and frozen calibration
mappings are ready. It does **not** establish independent calibration or the final novelty
claim, and it does not authorize test inference; those require a separate readiness gate.
